# Notebook F: Q&A Qualitative Evaluation

**Goal**: Load the fine-tuned QLoRA adapter weights (`spurgeon_lora_qa`) on top of the base model, switch it to fast inference mode, and run a test battery of diverse theological and edge-case questions to verify quality and formatting compliance.

## 0. Installation

This notebook requires **Unsloth** and its dependencies (for fast 4-bit inference with QLoRA adapters). 
The installation is optimized for Kaggle, Google Colab, and local CUDA environments.


In [2]:
%%capture
import os

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = "COLAB_GPU" in os.environ

print("🔧 Installing Unsloth and dependencies...")

if IS_KAGGLE or IS_COLAB:
    # Best for Kaggle/Colab: installs latest Unsloth with compatible torch/xformers/bitsandbytes
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes
else:
    # Local install (user should have CUDA-enabled PyTorch already)
    !pip install unsloth
    !pip install --no-deps xformers peft accelerate bitsandbytes trl

print("✅ Installation complete!")
print("   If you see version conflicts or are in Colab/Kaggle, restart the runtime/kernel now.")
print("   Then re-run this cell and continue.")


## 1. Setup & Environment Setup

In [ ]:
import os

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = "COLAB_GPU" in os.environ

if IS_KAGGLE:
    BASE_MODEL_NAME = "/kaggle/input/datasets/rafaelvieira1/spurgeon-lora-final/spurgeon_f16_gguf"
    ADAPTER_DIR = "/kaggle/input/datasets/rafaelvieira1/spurgeon-fine-tuning-trained/spurgeon_lora_qa"
    GGUF_EXPORT_DIR = "/kaggle/working/spurgeon_qa_f16_gguf"
elif IS_COLAB:
    BASE_MODEL_NAME = "/content/spurgeon_phase1_merged"
    ADAPTER_DIR = "/content/spurgeon_lora_qa"
    GGUF_EXPORT_DIR = "/content/spurgeon_qa_f16_gguf"
else:
    BASE_MODEL_NAME = "../models/spurgeon_phase1_merged"
    ADAPTER_DIR = "../models/spurgeon_lora_qa"
    GGUF_EXPORT_DIR = "../models/spurgeon_qa_f16_gguf"

print(f"Base Model: {BASE_MODEL_NAME}")
print(f"Adapter Dir: {ADAPTER_DIR}")
print(f"GGUF Export Dir: {GGUF_EXPORT_DIR}")

Base Model: /kaggle/input/datasets/rafaelvieira1/spurgeon-lora-final/spurgeon_f16_gguf
Adapter Dir: /kaggle/input/datasets/rafaelvieira1/spurgeon-fine-tuning-trained/spurgeon_lora_qa


## 2. Load Model & Apply Adapter

In [ ]:
import torch
from unsloth import FastLanguageModel
from transformers import PreTrainedTokenizerFast
from unsloth.chat_templates import get_chat_template
import json
import shutil
import tempfile
import os

# Monkey-patch the incorrect Hugging Face mistral regex warning/patching bug for Qwen model
PreTrainedTokenizerFast._patch_mistral_regex = classmethod(lambda cls, tokenizer, *args, **kwargs: tokenizer)

MAX_SEQ_LENGTH = 2048

# Create a patched temporary adapter directory to force Unsloth to find the environment-correct base model path
temp_adapter_dir = os.path.join(tempfile.gettempdir(), "patched_adapter")
if os.path.exists(temp_adapter_dir):
    shutil.rmtree(temp_adapter_dir)
shutil.copytree(ADAPTER_DIR, temp_adapter_dir)

config_path = os.path.join(temp_adapter_dir, "adapter_config.json")
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config = json.load(f)
    config["base_model_name_or_path"] = BASE_MODEL_NAME
    with open(config_path, "w") as f:
        json.dump(config, f, indent=4)
    print(f"Patched adapter config to point to base model: {BASE_MODEL_NAME}")

# Load model and tokenizer directly with adapter overlay using the patched config path
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=temp_adapter_dir,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Apply the correct ChatML template to the tokenizer for Qwen2.5/ChatML compatibility
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

# Set model to fast inference mode
FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth 2026.6.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048, padding_idx=151665)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

## 3. Define Qualitative Test Battery

In [8]:
test_prompts = [
    # 1. Doctrinal
    {
        "type": "Doctrinal",
        "question": "What is Charles Spurgeon's understanding of the doctrine of election?",
        "context": "God hath chosen his people unto eternal life. It is not of man's will, nor of man's merit, but solely of the sovereign grace of the Almighty. Election is the foundation stone of our salvation, laid in eternity before the foundation of the world."
    },
    # 2. Expository
    {
        "type": "Expository",
        "question": "How does Spurgeon interpret the invitation 'Come unto me' in Matthew 11:28?",
        "context": "Christ calls the heavy-laden to come to Him. Coming to Him is not a physical movement, but the simple trust of the heart. He does not say 'go to the law' or 'go to your works', but 'come to Me', for in Him alone is rest."
    },
    # 3. Applicational
    {
        "type": "Applicational",
        "question": "What counsel does Spurgeon give to a Christian who feels God is distant?",
        "context": "When the clouds hide the sun, the sun is still there. If you feel God is distant, do not run to other comforts. Cling to His Word, trust Him in the dark, and say 'Though He slay me, yet will I trust Him.' Look to the Cross, not to your feelings."
    },
    # 4. Passage-based
    {
        "type": "Passage-based",
        "question": "What theological point is Spurgeon making about the blood covenant?",
        "context": "The blood is the life of the covenant. Without shedding of blood, there is no remission. It is the blood of the Lamb that seals the promise, and when the destroying angel sees the blood, he will pass over you. It is the only safety for the soul."
    },
    # 5. Biographical
    {
        "type": "Biographical",
        "question": "How did Spurgeon describe the moment of his own conversion?",
        "context": "I looked at Jesus, and the cloud was gone. I could have risen and sung with the warmest of them of the precious blood of Christ. The minister said 'Look to Jesus!', and I looked, and in that moment I was saved."
    },
    # 6. Edge Case (Out of Scope / Refusal)
    {
        "type": "Edge Case / Refusal",
        "question": "What did Spurgeon think about Charles Darwin's theory of evolution?",
        "context": "In my preaching on the sovereignty of God, I often declare: 'He is the Creator of all things, and He holds the stars in His hands. The works of His hands are excellent, and His law is perfect.'"
    }
]

## 4. Run Inference & Verify Style & Grounding

In [9]:
from transformers import TextStreamer

messages_template = [
    {
        "role": "system",
        "content": "You are Charles Haddon Spurgeon. Answer using only the information in the provided CONTEXT. Stay very close to the actual text."
    },
    {
        "role": "user",
        "content": "CONTEXT:\n{context}\n\nQUESTION:\n{question}"
    }
]

text_streamer = TextStreamer(tokenizer, skip_prompt=True)

for item in test_prompts:
    print(f"\n{'='*80}")
    print(f"TYPE: {item['type']}")
    print(f"QUESTION: {item['question']}")
    print(f"CONTEXT: {item['context']}")
    print(f"{'='*80}")
    
    # Format chat inputs
    user_content = f"CONTEXT:\n{item['context']}\n\nQUESTION:\n{item['question']}"
    messages = [
        messages_template[0],
        {"role": "user", "content": user_content}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    print("\n--- GENERATED RESPONSE ---")
    _ = model.generate(
        input_ids=inputs,
        streamer=text_streamer,
        max_new_tokens=256,
        use_cache=True,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=1.15,
    )
    print()


TYPE: Doctrinal
QUESTION: What is Charles Spurgeon's understanding of the doctrine of election?
CONTEXT: God hath chosen his people unto eternal life. It is not of man's will, nor of man's merit, but solely of the sovereign grace of the Almighty. Election is the foundation stone of our salvation, laid in eternity before the foundation of the world.


ValueError: Cannot use chat template functions because tokenizer.chat_template is not set and no template argument was passed! For information about writing templates and setting the tokenizer.chat_template attribute, please see the documentation at https://huggingface.co/docs/transformers/main/en/chat_templating

## 5. Convert and Export to GGUF (Original 16-bit)

We merge our QA fine-tuned PEFT adapter weights back into the base model and export directly to GGUF format. We specify `quantization_method = "f16"` to preserve the original 16-bit floating point precision of the weights without any quantization loss.

In [ ]:
# Convert to GGUF format locally/Google Colab/Kaggle. 
# Unsloth will compile/download llama.cpp internally and output the merged F16 GGUF file.
model.save_pretrained_gguf(
    GGUF_EXPORT_DIR,
    tokenizer,
    quantization_method = "f16",
)

## 6. Upload GGUF Model to Hugging Face Hub

We securely load your Hugging Face write token from Kaggle Secrets, initialize the Hugging Face API, and upload the merged `spurgeon_qa_f16_gguf` file matching the `*.gguf` pattern directly to your model repository under `qwen2.5-3b-spurgeon-qa-gguf-phase2`.

In [ ]:
# Upload GGUF file to Hugging Face Hub
from huggingface_hub import HfApi, login
import glob
import os

try:
    # Load Hugging Face API write token with fallbacks per environment (Kaggle Secrets, Colab Secrets, Env Vars)
    hf_token = None
    if IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            user_secrets = UserSecretsClient()
            hf_token = user_secrets.get_secret("HF_TOKEN")
        except Exception:
            pass
            
    if not hf_token and IS_COLAB:
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
        except Exception:
            pass
            
    if not hf_token:
        hf_token = os.environ.get("HF_TOKEN")
        
    if not hf_token:
        raise ValueError("HF_TOKEN write token not found in Kaggle Secrets, Google Colab userdata, or project environment variables.")
        
    login(token=hf_token)
    
    api = HfApi()
    repo_id = "rafaelvieirar1r/qwen2.5-3b-spurgeon-qa-gguf-phase2"
    
    print(f"Creating Hugging Face repository {repo_id} (if it does not exist)...")
    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    
    # Dynamically find any .gguf file under the GGUF export parent directory or fallback globally to working space
    search_dir = os.path.dirname(os.path.normpath(GGUF_EXPORT_DIR)) if 'GGUF_EXPORT_DIR' in globals() else "."
    if not os.path.exists(search_dir):
        search_dir = "."
        
    gguf_files = glob.glob(os.path.join(search_dir, "**/*.gguf"), recursive=True)
    if not gguf_files:
        # Global workspace fallback
        gguf_files = glob.glob("**/*.gguf", recursive=True)
        
    if not gguf_files:
        raise FileNotFoundError(f"No GGUF file found under {search_dir} or current workspace!")
    
    file_path = gguf_files[0]
    file_name = os.path.basename(file_path)

    print(f"Found GGUF file: {file_path}")
    print(f"Uploading {file_name} to Hugging Face Hub...")
    api.upload_file(
        path_or_fileobj=file_path,
        path_in_repo=file_name,
        repo_id=repo_id,
        repo_type="model"
    )
    print("Model successfully uploaded to Hugging Face!")
except Exception as e:
    print(f"Failed to upload to Hugging Face: {e}")
    print("Please ensure you have toggled Internet ON and added the 'HF_TOKEN' key to your secrets or environment variables.")